In [ ]:
import time
import sys

# Aumenta o limite de recursão para safety em tabuleiros grandes
sys.setrecursionlimit(10000)

class ReguaPuzzleBacktracking:
    def __init__(self, estado_inicial):
        self.estado_inicial = tuple(estado_inicial)
        self.N = (len(estado_inicial) - 1) // 2

        # Estatísticas globais
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

        # Controle de histórico para evitar loops no caminho atual
        self.visitados_caminho_atual = set()

        # Melhor solução encontrada (caminho, custo)
        self.melhor_caminho = None
        self.melhor_custo = float('inf')

    def eh_meta(self, estado):
        """
        Verifica se o estado é uma meta: nenhum 'B' pode estar à direita de qualquer 'A'.
        Ou seja, se encontrarmos um 'A', não pode haver nenhum 'B' depois dele.
        """
        encontrou_A = False
        for char in estado:
            if char == 'A':
                encontrou_A = True
            elif char == 'B' and encontrou_A:
                return False
        return True

    def obter_sucessores(self, estado):
        """
        Gera os estados sucessores válidos e os custos dos movimentos.
        Regra: distância máxima de N posições até o espaço vazio '_'.
        """
        sucessores = []
        idx_vazio = estado.index('_')
        tamanho = len(estado)

        # Verifica todas as posições possíveis na régua
        for idx_bloco in range(tamanho):
            if idx_bloco == idx_vazio:
                continue

            distancia = abs(idx_vazio - idx_bloco)
            if distancia <= self.N:
                # Gerar o novo estado trocando o bloco com o vazio
                novo_estado = list(estado)
                novo_estado[idx_vazio], novo_estado[idx_bloco] = novo_estado[idx_bloco], novo_estado[idx_vazio]
                sucessores.append((tuple(novo_estado), distancia))

        return sucessores

    def buscar(self):
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0
        self.melhor_caminho = None
        self.melhor_custo = float('inf')
        self.visitados_caminho_atual = set()

        tempo_inicio = time.time()

        # Dispara a busca recursiva a partir do estado inicial
        self._backtracking_recursivo(self.estado_inicial, [self.estado_inicial], 0)

        tempo_fim = time.time()
        tempo_execucao = tempo_fim - tempo_inicio

        # Cálculo do fator de ramificação médio
        fator_ramificacao_medio = (self.total_filhos_gerados / self.nos_expandidos) if self.nos_expandidos > 0 else 0

        return {
            "caminho": self.melhor_caminho,
            "profundidade": len(self.melhor_caminho) - 1 if self.melhor_caminho else 0,
            "custo": self.melhor_custo if self.melhor_caminho else None,
            "nos_expandidos": self.nos_expandidos,
            "nos_visitados": self.nos_visitados,
            "fator_ramificacao_medio": fator_ramificacao_medio,
            "tempo_execucao": tempo_execucao
        }

    def _backtracking_recursivo(self, estado_atual, caminho_atual, custo_atual):
        self.nos_visitados += 1

        # Se o custo atual já for pior ou igual à melhor solução achada, podamos o ramo
        if custo_atual >= self.melhor_custo:
            return

        # Se atingiu o objetivo, atualizamos a melhor solução
        if self.eh_meta(estado_atual):
            self.melhor_custo = custo_atual
            self.melhor_caminho = list(caminho_atual)
            return

        # Expansão do nó
        self.nos_expandidos += 1
        sucessores = self.obter_sucessores(estado_atual)
        self.total_filhos_gerados += len(sucessores)

        # Adiciona o estado atual ao conjunto do caminho para evitar ciclos
        self.visitados_caminho_atual.add(estado_atual)

        # Ordenar sucessores pelo menor custo de pulo pode ajudar a achar soluções boas mais rápido (heurística de poda)
        sucessores.sort(key=lambda x: x[1])

        for proximo_estado, custo_movimento in sucessores:
            if proximo_estado not in self.visitados_caminho_atual:
                # Chamada recursiva (avanço)
                self._backtracking_recursivo(
                    proximo_estado,
                    caminho_atual + [proximo_estado],
                    custo_atual + custo_movimento
                )

        # Backtrack: remove do conjunto do caminho ao desempilhar a recursão
        self.visitados_caminho_atual.remove(estado_atual)

# --- Exemplo de Execução baseado no enunciado (N=2) ---
if __name__ == "__main__":
    # Estado inicial sugerido no PDF para N=2: 'B', 'A', '_', 'A', 'B'
    estado_inicial_teste = ['B', 'A', '_', 'A', 'B']

    print(f"Iniciando Busca Backtracking para o Estado Inicial: {estado_inicial_teste}\n")

    puzzle = ReguaPuzzleBacktracking(estado_inicial_teste)
    resultado = puzzle.buscar()

    # Exibição formatada das Estatísticas
    print("==================================================")
    print("            PROPRIEDADES DA SOLUÇÃO               ")
    print("==================================================")
    if resultado["caminho"]:
        print("Caminho encontrado:")
        for i, est in enumerate(resultado["caminho"]):
            print(f"  Passo {i}: {' '.join(est)}")
        print(f"\nProfundidade da Solução : {resultado['profundidade']}")
        print(f"Custo total da Solução  : {resultado['custo']}")
    else:
        print("Nenhuma solução encontrada.")

    print("--------------------------------------------------")
    print(f"Total de nós visitados        : {resultado['nos_visitados']}")
    print(f"Total de nós expandidos       : {resultado['nos_expandidos']}")
    print(f"Fator de ramificação médio     : {resultado['fator_ramificacao_medio']:.2f}")
    print(f"Tempo de execução             : {resultado['tempo_execucao']:.6f} segundos")
    print("==================================================")

Iniciando Busca Backtracking para o Estado Inicial: ['B', 'A', '_', 'A', 'B']

            PROPRIEDADES DA SOLUÇÃO               
Caminho encontrado:
  Passo 0: B A _ A B
  Passo 1: B A B A _
  Passo 2: B A B _ A
  Passo 3: B _ B A A

Profundidade da Solução : 3
Custo total da Solução  : 5
--------------------------------------------------
Total de nós visitados        : 238
Total de nós expandidos       : 162
Fator de ramificação médio     : 2.72
Tempo de execução             : 0.000563 segundos


In [ ]:
import time
from collections import deque

class ReguaPuzzleBFS:
    def __init__(self, estado_inicial):
        self.estado_inicial = tuple(estado_inicial)
        self.N = (len(estado_inicial) - 1) // 2

        # Estatísticas globais
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

    def eh_meta(self, estado):
        """
        Verifica se o estado é uma meta: nenhum 'B' pode estar à direita de qualquer 'A'.
        """
        encontrou_A = False
        for char in estado:
            if char == 'A':
                encontrou_A = True
            elif char == 'B' and encontrou_A:
                return False
        return True

    def obter_sucessores(self, estado):
        """
        Gera os estados sucessores válidos e os custos dos movimentos.
        Regra: distância máxima de N posições até o espaço vazio '_'.
        """
        sucessores = []
        idx_vazio = estado.index('_')
        tamanho = len(estado)

        for idx_bloco in range(tamanho):
            if idx_bloco == idx_vazio:
                continue

            distancia = abs(idx_vazio - idx_bloco)
            if distancia <= self.N:
                novo_estado = list(estado)
                novo_estado[idx_vazio], novo_estado[idx_bloco] = novo_estado[idx_bloco], novo_estado[idx_vazio]
                sucessores.append((tuple(novo_estado), distancia))

        return sucessores

    def buscar(self):
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

        tempo_inicio = time.time()

        # Conjunto global de visitados para evitar loops de repetição de estados
        visitados = set()

        # Fila para o BFS armazenando: (estado_atual, caminho_ate_aqui, custo_acumulado)
        fila = deque([(self.estado_inicial, [self.estado_inicial], 0)])
        visitados.add(self.estado_inicial)
        self.nos_visitados += 1

        solucao = None

        while fila:
            estado_atual, caminho_atual, custo_atual = fila.popleft()

            # Se atingiu o objetivo, encerra (BFS garante o caminho mais curto em passos)
            if self.eh_meta(estado_atual):
                solucao = {
                    "caminho": caminho_atual,
                    "profundidade": len(caminho_atual) - 1,
                    "custo": custo_atual
                }
                break

            # Expansão do nó
            self.nos_expandidos += 1
            sucessores = self.obter_sucessores(estado_atual)
            self.total_filhos_gerados += len(sucessores)

            for proximo_estado, custo_movimento in sucessores:
                if proximo_estado not in visitados:
                    visitados.add(proximo_estado)
                    self.nos_visitados += 1

                    fila.append((
                        proximo_estado,
                        caminho_atual + [proximo_estado],
                        custo_atual + custo_movimento
                    ))

        tempo_fim = time.time()
        tempo_execucao = tempo_fim - tempo_inicio

        fator_ramificacao_medio = (self.total_filhos_gerados / self.nos_expandidos) if self.nos_expandidos > 0 else 0

        if solucao:
            solucao.update({
                "nos_expandidos": self.nos_expandidos,
                "nos_visitados": self.nos_visitados,
                "fator_ramificacao_medio": fator_ramificacao_medio,
                "tempo_execucao": tempo_execucao
            })
            return solucao
        else:
            return {
                "caminho": None,
                "profundidade": 0,
                "custo": None,
                "nos_expandidos": self.nos_expandidos,
                "nos_visitados": self.nos_visitados,
                "fator_ramificacao_medio": fator_ramificacao_medio,
                "tempo_execucao": tempo_execucao
            }

# --- Exemplo de Execução baseado no enunciado (N=2) ---
if __name__ == "__main__":
    # Estado inicial sugerido no PDF para N=2: 'B', 'A', '_', 'A', 'B' [cite: 17, 18]
    estado_inicial_teste = ['B', 'A', '_', 'A', 'B']

    print(f"Iniciando Busca em Largura (BFS) para o Estado Inicial: {estado_inicial_teste}\n")

    puzzle = ReguaPuzzleBFS(estado_inicial_teste)
    resultado = puzzle.buscar()

    # Exibição formatada das Estatísticas
    print("==================================================")
    print("            PROPRIEDADES DA SOLUÇÃO (BFS)         ")
    print("==================================================")
    if resultado["caminho"]:
        print("Caminho encontrado:")
        for i, est in enumerate(resultado["caminho"]):
            print(f"  Passo {i}: {' '.join(est)}")
        print(f"\nProfundidade da Solução : {resultado['profundidade']}")
        print(f"Custo total da Solução  : {resultado['custo']}")
    else:
        print("Nenhuma solução encontrada.")

    print("--------------------------------------------------")
    print(f"Total de nós visitados        : {resultado['nos_visitados']}")
    print(f"Total de nós expandidos       : {resultado['nos_expandidos']}")
    print(f"Fator de ramificação médio     : {resultado['fator_ramificacao_medio']:.2f}")
    print(f"Tempo de execução             : {resultado['tempo_execucao']:.6f} segundos")
    print("==================================================")

Iniciando Busca em Largura (BFS) para o Estado Inicial: ['B', 'A', '_', 'A', 'B']

            PROPRIEDADES DA SOLUÇÃO (BFS)         
Caminho encontrado:
  Passo 0: B A _ A B
  Passo 1: B A B A _
  Passo 2: B A B _ A
  Passo 3: B _ B A A

Profundidade da Solução : 3
Custo total da Solução  : 5
--------------------------------------------------
Total de nós visitados        : 19
Total de nós expandidos       : 12
Fator de ramificação médio     : 2.92
Tempo de execução             : 0.000068 segundos


In [ ]:
import time

class ReguaPuzzleDLS:
    def __init__(self, estado_inicial):
        self.estado_inicial = tuple(estado_inicial)
        self.N = (len(estado_inicial) - 1) // 2

        # Estatísticas globais
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

    def eh_meta(self, estado):
        """
        Verifica se o estado é uma meta: nenhum 'B' pode estar à direita de qualquer 'A'.
        """
        encontrou_A = False
        for char in estado:
            if char == 'A':
                encontrou_A = True
            elif char == 'B' and encontrou_A:
                return False
        return True

    def obter_sucessores(self, estado):
        """
        Gera os estados sucessores válidos e os custos dos movimentos.
        Regra: distância máxima de N posições até o espaço vazio '_'.
        """
        sucessores = []
        idx_vazio = estado.index('_')
        tamanho = len(estado)

        for idx_bloco in range(tamanho):
            if idx_bloco == idx_vazio:
                continue

            distancia = abs(idx_vazio - idx_bloco)
            if distancia <= self.N:
                novo_estado = list(estado)
                novo_estado[idx_vazio], novo_estado[idx_bloco] = novo_estado[idx_bloco], novo_estado[idx_vazio]
                sucessores.append((tuple(novo_estado), distancia))

        return sucessores

    def buscar(self, limite_profundidade):
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

        tempo_inicio = time.time()

        # Para evitar ciclos dentro do mesmo ramo ativo do caminho
        caminho_atual = [self.estado_inicial]
        visitados_ramo = {self.estado_inicial}

        # Executa a busca recursiva
        solucao = self._dls_recursivo(
            self.estado_inicial,
            limite_profundidade,
            caminho_atual,
            visitados_ramo,
            custo_atual=0
        )

        tempo_fim = time.time()
        tempo_execucao = tempo_fim - tempo_inicio

        fator_ramificacao_medio = (self.total_filhos_gerados / self.nos_expandidos) if self.nos_expandidos > 0 else 0

        resultado = {
            "nos_expandidos": self.nos_expandidos,
            "nos_visitados": self.nos_visitados,
            "fator_ramificacao_medio": fator_ramificacao_medio,
            "tempo_execucao": tempo_execucao
        }

        if solucao:
            resultado.update(solucao)
        else:
            resultado.update({"caminho": None, "profundidade": 0, "custo": None})

        return resultado

    def _dls_recursivo(self, estado_atual, limite, caminho, visitados_ramo, custo_atual):
        self.nos_visitados += 1

        # Se atingiu a meta, retorna o resultado imediatamente
        if self.eh_meta(estado_atual):
            return {
                "caminho": list(caminho),
                "profundidade": len(caminho) - 1,
                "custo": custo_atual
            }

        # Se o limite chegou a 0 e não é meta, corta a busca (cutoff)
        if limite <= 0:
            return None

        # Expansão do nó
        self.nos_expandidos += 1
        sucessores = self.obter_sucessores(estado_atual)
        self.total_filhos_gerados += len(sucessores)

        for proximo_estado, custo_movimento in sucessores:
            if proximo_estado not in visitados_ramo:
                # Avanço
                visitados_ramo.add(proximo_estado)
                caminho.append(proximo_estado)

                resultado_filho = self._dls_recursivo(
                    proximo_estado,
                    limite - 1,
                    caminho,
                    visitados_ramo,
                    custo_atual + custo_movimento
                )

                if resultado_filho:
                    return resultado_filho  # Retorna se encontrou a solução

                # Backtracking (remover do caminho para testar alternativas)
                caminho.pop()
                visitados_ramo.remove(proximo_estado)

        return None

# --- Exemplo de Execução baseado no enunciado (N=2) ---
if __name__ == "__main__":
    # Estado inicial sugerido no PDF para N=2: 'B', 'A', '_', 'A', 'B'
    estado_inicial_teste = ['B', 'A', '_', 'A', 'B']

    # Definindo um limite arbitrário (ex: 10 passos)
    limite = 10

    print(f"Iniciando Busca em Profundidade Limitada (DLS) com Limite = {limite}")
    print(f"Estado Inicial: {estado_inicial_teste}\n")

    puzzle = ReguaPuzzleDLS(estado_inicial_teste)
    resultado = puzzle.buscar(limite_profundidade=limite)

    # Exibição formatada das Estatísticas
    print("==================================================")
    print("            PROPRIEDADES DA SOLUÇÃO (DLS)         ")
    print("==================================================")
    if resultado["caminho"]:
        print("Caminho encontrado:")
        for i, est in enumerate(resultado["caminho"]):
            print(f"  Passo {i}: {' '.join(est)}")
        print(f"\nProfundidade da Solução : {resultado['profundidade']}")
        print(f"Custo total da Solução  : {resultado['custo']}")
    else:
        print(f"Nenhuma solução encontrada dentro do limite {limite}.")

    print("--------------------------------------------------")
    print(f"Total de nós visitados        : {resultado['nos_visitados']}")
    print(f"Total de nós expandidos       : {resultado['nos_expandidos']}")
    print(f"Fator de ramificação médio     : {resultado['fator_ramificacao_medio']:.2f}")
    print(f"Tempo de execução             : {resultado['tempo_execucao']:.6f} segundos")
    print("==================================================")

Iniciando Busca em Profundidade Limitada (DLS) com Limite = 10
Estado Inicial: ['B', 'A', '_', 'A', 'B']

            PROPRIEDADES DA SOLUÇÃO (DLS)         
Caminho encontrado:
  Passo 0: B A _ A B
  Passo 1: _ A B A B
  Passo 2: A _ B A B
  Passo 3: A B _ A B
  Passo 4: _ B A A B
  Passo 5: B _ A A B
  Passo 6: B A A _ B
  Passo 7: B A A B _
  Passo 8: B A _ B A
  Passo 9: B _ A B A
  Passo 10: B B A _ A

Profundidade da Solução : 10
Custo total da Solução  : 15
--------------------------------------------------
Total de nós visitados        : 14
Total de nós expandidos       : 11
Fator de ramificação médio     : 2.91
Tempo de execução             : 0.000061 segundos


In [ ]:
import time
import heapq

class ReguaPuzzleBuscaOrdenada:
    def __init__(self, estado_inicial):
        self.estado_inicial = tuple(estado_inicial)
        self.N = (len(estado_inicial) - 1) // 2

        # Estatísticas globais
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

    def eh_meta(self, estado):
        """
        Verifica se o estado é uma meta: nenhum 'B' pode estar à direita de qualquer 'A'[cite: 8].
        """
        encontrou_A = False
        for char in estado:
            if char == 'A':
                encontrou_A = True
            elif char == 'B' and encontrou_A:
                return False
        return True

    def obter_sucessores(self, estado):
        """
        Gera os estados sucessores válidos e os custos dos movimentos.
        Regra: distância máxima de N posições até o espaço vazio '_'[cite: 12].
        O custo é igual à distância do pulo.
        """
        sucessores = []
        idx_vazio = estado.index('_')
        tamanho = len(estado)

        for idx_bloco in range(tamanho):
            if idx_bloco == idx_vazio:
                continue

            distancia = abs(idx_vazio - idx_bloco)
            if distancia <= self.N: # [cite: 12]
                novo_estado = list(estado)
                novo_estado[idx_vazio], novo_estado[idx_bloco] = novo_estado[idx_bloco], novo_estado[idx_vazio]
                sucessores.append((tuple(novo_estado), distancia)) #

        return sucessores

    def buscar(self):
        self.nos_expandidos = 0
        self.nos_visitados = 0
        self.total_filhos_gerados = 0

        tempo_inicio = time.time()

        # Dicionário de fechados: armazena o menor custo conhecido para chegar a cada estado
        # Isso evita reexpandir nós por caminhos mais caros
        custos_otimos = {}

        # Fila de prioridades (Heap). Formato dos elementos: (custo_acumulado, id_unico, estado_atual, caminho)
        # O id_unico ajuda o heapq a desempatar sem tentar comparar as strings/tuplas do estado diretamente.
        fila_prioridade = []
        id_contador = 0

        # Insere o nó inicial
        heapq.heappush(fila_prioridade, (0, id_contador, self.estado_inicial, [self.estado_inicial]))
        custos_otimos[self.estado_inicial] = 0
        self.nos_visitados += 1

        solucao = None

        while fila_prioridade:
            custo_atual, _, estado_atual, caminho_atual = heapq.heappop(fila_prioridade)

            # Se o nó retirado já foi alcançado antes por um caminho mais barato, desconsideramos
            if custo_atual > custos_otimos.get(estado_atual, float('inf')):
                continue

            # Na Busca Ordenada, o teste de meta é feito ao RETIRAR o nó da fila (garante otimalidade)
            if self.eh_meta(estado_atual):
                solucao = {
                    "caminho": caminho_atual,
                    "profundidade": len(caminho_atual) - 1,
                    "custo": custo_atual
                }
                break

            # Expansão do nó
            self.nos_expandidos += 1
            sucessores = self.obter_sucessores(estado_atual)
            self.total_filhos_gerados += len(sucessores)

            for proximo_estado, custo_movimento in sucessores:
                novo_custo = custo_atual + custo_movimento

                # Se o estado nunca foi visitado OU se encontramos um caminho mais barato para ele
                if proximo_estado not in custos_otimos or novo_custo < custos_otimos[proximo_estado]:
                    custos_otimos[proximo_estado] = novo_custo
                    self.nos_visitados += 1
                    id_contador += 1

                    heapq.heappush(fila_prioridade, (
                        novo_custo,
                        id_contador,
                        proximo_estado,
                        caminho_atual + [proximo_estado]
                    ))

        tempo_fim = time.time()
        tempo_execucao = tempo_fim - tempo_inicio

        fator_ramificacao_medio = (self.total_filhos_gerados / self.nos_expandidos) if self.nos_expandidos > 0 else 0

        if solucao:
            solucao.update({
                "nos_expandidos": self.nos_expandidos,
                "nos_visitados": self.nos_visitados,
                "fator_ramificacao_medio": fator_ramificacao_medio,
                "tempo_execucao": tempo_execucao
            })
            return solucao
        else:
            return {
                "caminho": None,
                "profundidade": 0,
                "custo": None,
                "nos_expandidos": self.nos_expandidos,
                "nos_visitados": self.nos_visitados,
                "fator_ramificacao_medio": fator_ramificacao_medio,
                "tempo_execucao": tempo_execucao
            }

# --- Exemplo de Execução baseado no enunciado (N=2) ---
if __name__ == "__main__":
    # Estado inicial sugerido no PDF para N=2: 'B', 'A', '_', 'A', 'B' [cite: 15, 16, 17, 18]
    estado_inicial_teste = ['B', 'A', '_', 'A', 'B']

    print(f"Iniciando Busca Ordenada (Custo Uniforme) para o Estado Inicial: {estado_inicial_teste}\n")

    puzzle = ReguaPuzzleBuscaOrdenada(estado_inicial_teste)
    resultado = puzzle.buscar()

    # Exibição formatada das Estatísticas
    print("==================================================")
    print("         PROPRIEDADES DA SOLUÇÃO (ORDENADA)       ")
    print("==================================================")
    if resultado["caminho"]:
        print("Caminho encontrado:")
        for i, est in enumerate(resultado["caminho"]):
            print(f"  Passo {i}: {' '.join(est)}")
        print(f"\nProfundidade da Solução : {resultado['profundidade']}")
        print(f"Custo total da Solução  : {resultado['custo']}") #
    else:
        print("Nenhuma solução encontrada.")

    print("--------------------------------------------------")
    print(f"Total de nós visitados        : {resultado['nos_visitados']}")
    print(f"Total de nós expandidos       : {resultado['nos_expandidos']}")
    print(f"Fator de ramificação médio     : {resultado['fator_ramificacao_medio']:.2f}")
    print(f"Tempo de execução             : {resultado['tempo_execucao']:.6f} segundos")
    print("==================================================")

Iniciando Busca Ordenada (Custo Uniforme) para o Estado Inicial: ['B', 'A', '_', 'A', 'B']

         PROPRIEDADES DA SOLUÇÃO (ORDENADA)       
Caminho encontrado:
  Passo 0: B A _ A B
  Passo 1: B A B A _
  Passo 2: B A B _ A
  Passo 3: B _ B A A

Profundidade da Solução : 3
Custo total da Solução  : 5
--------------------------------------------------
Total de nós visitados        : 19
Total de nós expandidos       : 12
Fator de ramificação médio     : 2.92
Tempo de execução             : 0.000089 segundos
